# ProtCell clustering and integration in two published studies

Run the **Sacco et al. (2022)** workflow first, including cluster-marker analysis; then run the **Beneyto-Calabuig et al. (2023)** workflow. Each study has explicit input paths, metadata columns, adaptation parameters, checkpoints and output directories.

| Published study | Processed protein file | Paper panels |
| --- | --- | --- |
| Sacco et al. (2022), *Immunopathological signatures in multisystem inflammatory syndrome in children and pediatric COVID-19* (Nature Medicine) | `CLR_data/Dataset74.h5ad` | Supplementary Fig. 2b (integration), 2c (cell-type/batch UMAPs), 2d (cluster-marker contrasts) |
| Beneyto-Calabuig et al. (2023), *Clonally resolved single-cell multi-omics identifies routes of cellular differentiation in acute myeloid leukemia* (Cell Stem Cell) | `CLR_data/Dataset107.h5ad` | Fig. 2a and Supplementary Fig. 2b (integration) |

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path(os.environ.get("PROTCELL_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (PROJECT_ROOT / "train.py").is_file() and (PROJECT_ROOT.parent / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "train.py").is_file(), "Set PROTCELL_PROJECT_ROOT to your ProtCell clone before starting JupyterLab."
OUTPUT_ROOT = Path(os.environ.get("PROTCELL_OUTPUT_DIR", str(PROJECT_ROOT / "results/outputs")))
OUTPUT_DIR = OUTPUT_ROOT / "cluster"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.chdir(OUTPUT_DIR) # trainer logs and new files stay with this experiment

old_cache_settings = {key: os.environ.get(key) for key in ["MPLCONFIGDIR", "NUMBA_CACHE_DIR", "XDG_CACHE_HOME"]}
plot_cache = tempfile.TemporaryDirectory(prefix="protcell-cluster-", dir=OUTPUT_DIR)
os.environ["MPLCONFIGDIR"] = plot_cache.name
os.environ["NUMBA_CACHE_DIR"] = str(Path(plot_cache.name) / "numba")
os.environ["XDG_CACHE_HOME"] = str(Path(plot_cache.name) / "xdg")

import json
import hashlib
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse, stats
from tqdm.auto import tqdm
from IPython.display import display
from model.model import ProteinLanguageModel
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from finetuner.BatchCorrection import BatchCorrectionFineTuner
from model import utils
# Use the R installation bundled with the active conda environment for kBET.
r_home = Path(sys.prefix) / "lib/R"
if r_home.is_dir():
    os.environ.setdefault("R_HOME", str(r_home))
import scib
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif"})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
UMAP_SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_ROOT = Path(os.environ.get("PROTCELL_DATA_ROOT", str(PROJECT_ROOT / "data"))).expanduser().resolve()
BASE_MODEL_PATH = PROJECT_ROOT / "checkpoints/basemodel/large-scale-new-10/model_369999.pt"
print("Device:", DEVICE)
print("Outputs:", OUTPUT_DIR)

## 1. Sacco et al. (2022) — load protein measurements

Read the CLR-processed `Dataset74.h5ad` used for this study. `Batch` defines experimental batches/cohorts and `coarsecelltype` supplies reference cell types for evaluation. Integration adapts to the full dataset; reference labels determine the evaluation cluster count and assess biological organization.

Outputs for this study are written under `outputs/cluster/Sacco2022/`.

In [ ]:
SACCO_OUTPUT_DIR = OUTPUT_DIR / "Sacco2022"
SACCO_FIGURE_DIR = SACCO_OUTPUT_DIR / "figures"
SACCO_RESULT_DIR = SACCO_OUTPUT_DIR / "results"
SACCO_MODEL_DIR = SACCO_OUTPUT_DIR / "models"
for directory in [SACCO_FIGURE_DIR, SACCO_RESULT_DIR, SACCO_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sacco_data = ad.read_h5ad(DATA_ROOT / "CLR_data/Dataset74.h5ad")
SACCO_BATCH_COLUMN = "Batch"
SACCO_CELLTYPE_COLUMN = "coarsecelltype"
for column in [SACCO_BATCH_COLUMN, SACCO_CELLTYPE_COLUMN]:
    sacco_data.obs[column] = sacco_data.obs[column].astype(str).replace({"nan": "unknown", "None": "unknown"})
display(sacco_data)
display(sacco_data.obs.groupby(SACCO_BATCH_COLUMN, observed=True).size())

RUN_SACCO_ADAPTATION = True
RUN_SACCO_EMBEDDINGS = True
RUN_SACCO_EVALUATION = True
RUN_SACCO_UMAP_PLOTS = True

SACCO_ADAPTED_MODEL_PATH = PROJECT_ROOT / "checkpoints/BatCorr/74/finetune_5.pt"
RUN_SACCO_MARKER_ANALYSIS = True
SACCO_CLUSTER_CONTRASTS = [("0", "15"), ("0", "13"), ("1", "3")]


### Sacco et al. (2022) — load pretrained ProtCell

Use the projected CLS output for cell representations. Each study starts from the same pretrained checkpoint.

In [ ]:
sacco_model_path = BASE_MODEL_PATH
sacco_model = ProteinLanguageModel.load_model_from_checkpoint(str(sacco_model_path), device=DEVICE).to(DEVICE)
sacco_model.eval()
display(sacco_model)

### Sacco et al. (2022) — fine-tune the integration adapter

Fine-tune the integration adapter for five epochs. Use the resulting checkpoint for embedding extraction, evaluation and marker analysis. To reuse a saved adapter, set RUN_SACCO_ADAPTATION=False and specify SACCO_ADAPTED_MODEL_PATH.

In [ ]:
if RUN_SACCO_ADAPTATION:
    sacco_adapter_model = ProteinLanguageModel.load_model_from_checkpoint(
        str(BASE_MODEL_PATH), device=DEVICE
    ).to(DEVICE)
    sacco_integration_trainer = BatchCorrectionFineTuner(
        model=sacco_adapter_model,
        adata=sacco_data,
        device=DEVICE,
        batch_name=SACCO_BATCH_COLUMN,
        lr=1e-3,
        batch_size=512,
        expr_name="integration",
        trainable_modules={"lora": 1e-3, "reg_proj": 1e-5, "expr_head": 5e-5},
        temperature=0.5,
        reg=0.5,
        reg_m=0.5,
        tau_intra=0.05,
        lambda_intra=10,
    )
    sacco_integration_trainer.fit(
        num_epochs=5,
        save=True,
        save_dir=str(SACCO_MODEL_DIR),
        save_epoch_interval=5,
        early_stop=False,
        lora_rank=16,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["linear1", "linear2", "cls_proj.0", "cls_proj.3"],
    )
    SACCO_ADAPTED_MODEL_PATH = SACCO_MODEL_DIR / "integration" / "finetune_5.pt"
print("Adapted checkpoint:", SACCO_ADAPTED_MODEL_PATH)

### Sacco et al. (2022) — extract CLS embeddings and assign clusters

Extract zero-shot and adapted ProtCell representations separately. Record the model path and its SHA256 with the saved representation; cluster assignments and UMAP coordinates use that representation.

In [ ]:
sacco_representations = {}
if RUN_SACCO_EMBEDDINGS:
    sacco_representation_paths = {'zeroshot': BASE_MODEL_PATH}
    if SACCO_ADAPTED_MODEL_PATH is not None:
        sacco_representation_paths['finetune'] = Path(SACCO_ADAPTED_MODEL_PATH)
    for (sacco_method, sacco_checkpoint_path) in sacco_representation_paths.items():
        sacco_model = ProteinLanguageModel.load_model_from_checkpoint(str(sacco_checkpoint_path), device=DEVICE).to(DEVICE)
        sacco_model.eval()
        sacco_protein_ids = torch.tensor(sacco_model.tokenizer.encode(sacco_data.var_names.tolist()), device=DEVICE)
        sacco_x = sacco_data.X.toarray() if sparse.issparse(sacco_data.X) else np.asarray(sacco_data.X)
        sacco_embeddings = []
        with torch.no_grad():
            for sacco_start in tqdm(range(0, sacco_data.n_obs, 512), desc=sacco_method):
                sacco_expression = torch.as_tensor(sacco_x[sacco_start:sacco_start + 512], dtype=torch.float32, device=DEVICE)
                sacco_ids = sacco_protein_ids.unsqueeze(0).expand(len(sacco_expression), -1)
                sacco_embeddings.append(sacco_model.get_cell_embedding(input_ids=sacco_ids, expr_values=sacco_expression).cpu().numpy())
        sacco_result = sacco_data.copy()
        sacco_result.obsm['X_embed'] = np.vstack(sacco_embeddings)
        sacco_result = utils.run_leiden(sacco_result, n_cluster=sacco_result.obs[SACCO_CELLTYPE_COLUMN].nunique(), use_rep='X_embed', key_added='ProtCell_cluster')
        sc.tl.umap(sacco_result, random_state=UMAP_SEED)
        sacco_result.uns['ProtCell_checkpoint'] = str(sacco_checkpoint_path)
        sacco_result.uns['ProtCell_umap_seed'] = UMAP_SEED
        sacco_result.uns['ProtCell_dataset_author'] = "Sacco2022"
        sacco_result.uns['ProtCell_checkpoint_sha256'] = hashlib.sha256(Path(sacco_checkpoint_path).read_bytes()).hexdigest()
        sacco_result.write_h5ad(SACCO_RESULT_DIR / f'{sacco_method}.h5ad', compression='gzip')
        sacco_representations[sacco_method] = sacco_result

### Sacco et al. (2022) — UMAP by batch, cluster and reference cell type

The three colourings share the same saved coordinates. Save a triptych and individual maps; the cluster map labels cluster IDs on the point cloud.

In [ ]:
if RUN_SACCO_UMAP_PLOTS:
    assert RUN_SACCO_EMBEDDINGS and sacco_representations, "Run CLS extraction before plotting UMAPs."
    for sacco_method, sacco_representation in sacco_representations.items():
        sacco_map_data = sacco_representation.copy()
        assert 'X_umap' in sacco_map_data.obsm
        sacco_map_data.obs['ProtCell_cluster'] = pd.Categorical(sacco_map_data.obs['ProtCell_cluster'].astype(str))
        sacco_coordinates = sacco_map_data.obsm['X_umap'].copy()
        sacco_axes = sc.pl.umap(sacco_map_data, color=[SACCO_BATCH_COLUMN, 'ProtCell_cluster', SACCO_CELLTYPE_COLUMN], title=['Batch', 'ProtCell clusters', 'Reference cell type'], legend_loc='right margin', legend_fontsize=8, legend_fontoutline=2, na_in_legend=False, wspace=0.8, show=False)
        sacco_fig = sacco_axes[0].figure
        sacco_fig.savefig(SACCO_FIGURE_DIR / f'{sacco_method}_umap_triptych.pdf', bbox_inches='tight')
        sacco_fig.savefig(SACCO_FIGURE_DIR / f'{sacco_method}_umap_triptych.png', dpi=300, bbox_inches='tight')
        plt.show()
        for (sacco_key, sacco_label) in [(SACCO_BATCH_COLUMN, 'batch'), ('ProtCell_cluster', 'cluster'), (SACCO_CELLTYPE_COLUMN, 'celltype')]:
            sacco_ax = sc.pl.umap(sacco_map_data, color=sacco_key, title=f'ProtCell {sacco_method}: {sacco_label}', legend_loc='on data' if sacco_label == 'cluster' else 'right margin', legend_fontsize=8, legend_fontoutline=2, na_in_legend=False, show=False)
            sacco_fig = sacco_ax.figure
            sacco_fig.savefig(SACCO_FIGURE_DIR / f'{sacco_method}_umap_{sacco_label}.pdf', bbox_inches='tight')
            sacco_fig.savefig(SACCO_FIGURE_DIR / f'{sacco_method}_umap_{sacco_label}.png', dpi=300, bbox_inches='tight')
            plt.show()
        assert np.array_equal(sacco_map_data.obsm['X_umap'], sacco_coordinates)

### Sacco et al. (2022) — evaluate cell organization and batch mixing

Calculate ARI, NMI, raw cluster silhouette, scaled biological ASW, batch ASW, kBET, iLISI and cLISI. Compute the summary scores as follows: `avgBio=(ARI+NMI+Bio_ASW)/3`, `avgBatch=(iLISI+Batch_ASW)/2`, and `overall=0.6*avgBio+0.4*avgBatch`. The scaled kBET score is higher for better mixing; dependency failures are recorded separately.

In [ ]:
sacco_all_metrics = None
if RUN_SACCO_EVALUATION:
    assert RUN_SACCO_EMBEDDINGS and sacco_representations, "Run CLS extraction before evaluation."
    sacco_metric_rows = []
    sacco_metric_status = []
    for sacco_method, sacco_representation in sacco_representations.items():
        sacco_evaluation_data = sacco_representation.copy()
        sacco_expected_checkpoint = BASE_MODEL_PATH if sacco_method == "zeroshot" else SACCO_ADAPTED_MODEL_PATH
        assert Path(sacco_evaluation_data.uns["ProtCell_checkpoint"]).resolve() == Path(sacco_expected_checkpoint).resolve(), "Rerun embedding extraction after changing the checkpoint."
        assert sacco_evaluation_data.uns["ProtCell_checkpoint_sha256"] == hashlib.sha256(Path(sacco_expected_checkpoint).read_bytes()).hexdigest(), "Rerun embedding extraction after retraining."
        print(f'Evaluating {sacco_method}: {sacco_evaluation_data.n_obs:,} cells')
        sacco_ari = adjusted_rand_score(sacco_evaluation_data.obs[SACCO_CELLTYPE_COLUMN], sacco_evaluation_data.obs['ProtCell_cluster'])
        sacco_nmi = normalized_mutual_info_score(sacco_evaluation_data.obs[SACCO_CELLTYPE_COLUMN], sacco_evaluation_data.obs['ProtCell_cluster'])
        sacco_cluster_silhouette = silhouette_score(sacco_evaluation_data.obsm['X_embed'], sacco_evaluation_data.obs['ProtCell_cluster'].astype(str))
        sacco_bio_asw = scib.me.silhouette(sacco_evaluation_data, group_key=SACCO_CELLTYPE_COLUMN, embed='X_embed')
        sacco_batch_asw = scib.me.silhouette_batch(sacco_evaluation_data, batch_key=SACCO_BATCH_COLUMN, group_key=SACCO_CELLTYPE_COLUMN, embed='X_embed')
        try:
            sacco_kbet = scib.me.kBET(sacco_evaluation_data, batch_key=SACCO_BATCH_COLUMN, label_key=SACCO_CELLTYPE_COLUMN, type_='embed', embed='X_embed')
            sacco_metric_status.append(dict(method=sacco_method, metric='kBET', status='computed', detail=''))
        except (ModuleNotFoundError, ImportError, RuntimeError) as error:
            sacco_kbet = np.nan
            sacco_metric_status.append(dict(method=sacco_method, metric='kBET', status='unavailable', detail=str(error)))
            print('kBET could not be computed:', error)
        sacco_ilisi = scib.me.ilisi_graph(sacco_evaluation_data, batch_key=SACCO_BATCH_COLUMN, type_='embed', use_rep='X_embed')
        sacco_clisi = scib.me.clisi_graph(sacco_evaluation_data, label_key=SACCO_CELLTYPE_COLUMN, type_='embed', use_rep='X_embed')
        sacco_avgbio = (sacco_ari + sacco_nmi + sacco_bio_asw) / 3
        sacco_avgbatch = (sacco_ilisi + sacco_batch_asw) / 2
        sacco_metric_rows.append(dict(method=sacco_method, ARI=sacco_ari, NMI=sacco_nmi, Silhouette=sacco_cluster_silhouette, kBET=sacco_kbet, iLISI=sacco_ilisi, cLISI=sacco_clisi, Bio_ASW=sacco_bio_asw, Batch_ASW=sacco_batch_asw, avgBio=sacco_avgbio, avgBatch=sacco_avgbatch, overall=0.6 * sacco_avgbio + 0.4 * sacco_avgbatch))
    sacco_all_metrics = pd.DataFrame(sacco_metric_rows)
    sacco_all_metrics.to_csv(SACCO_RESULT_DIR / 'metrics.csv', index=False)
    pd.DataFrame(sacco_metric_status).to_csv(SACCO_RESULT_DIR / 'metric_status.csv', index=False)
    display(sacco_all_metrics)

### Sacco et al. (2022) — display all evaluation metrics

Display the complete table calculated above. The current cell representations and metrics stay in memory.


In [ ]:
if RUN_SACCO_EVALUATION:
    assert sacco_all_metrics is not None, "Run the evaluation cell before displaying metrics."
    sacco_protcell_metrics = sacco_all_metrics.copy()
    sacco_metric_columns = ['ARI', 'NMI', 'Silhouette', 'kBET', 'iLISI', 'cLISI', 'Bio_ASW', 'Batch_ASW', 'avgBio', 'avgBatch', 'overall']
    display(sacco_protcell_metrics[['method'] + sacco_metric_columns])
else:
    print("Enable RUN_SACCO_EVALUATION to calculate this study's metrics.")


### Sacco et al. (2022) — downstream cluster-marker analysis

Use the **same adapted Sacco representation evaluated above**. Display cluster composition by reference cell type before inspecting the reported 0 vs 15, 0 vs 13 and 1 vs 3 contrasts (Supplementary Fig. 2d). Numbered clusters from a new adaptation should be interpreted alongside this composition table.

In [ ]:
if RUN_SACCO_MARKER_ANALYSIS:
    assert RUN_SACCO_EMBEDDINGS and "finetune" in sacco_representations, "Extract the adapted Sacco embeddings before marker analysis."
    sacco_cluster_data = sacco_representations["finetune"].copy()
    assert Path(sacco_cluster_data.uns["ProtCell_checkpoint"]).resolve() == Path(SACCO_ADAPTED_MODEL_PATH).resolve(), "Rerun Sacco embedding extraction after changing the adapted checkpoint."
    assert sacco_cluster_data.uns["ProtCell_checkpoint_sha256"] == hashlib.sha256(Path(SACCO_ADAPTED_MODEL_PATH).read_bytes()).hexdigest(), "Rerun Sacco embedding extraction after retraining the model."
    sacco_cluster_data.obs["Nlouvain_embed"] = pd.Categorical(sacco_cluster_data.obs["ProtCell_cluster"].astype(str))
    sacco_cluster_composition = pd.crosstab(sacco_cluster_data.obs["Nlouvain_embed"], sacco_cluster_data.obs[SACCO_CELLTYPE_COLUMN], normalize="index")
    sacco_cluster_composition.to_csv(SACCO_RESULT_DIR / "cluster_celltype_composition.csv")
    display(sacco_cluster_composition)


### Sacco et al. (2022) — bidirectional marker functions

Rank each direction independently, retain adjusted P < 0.05 and positive fold changes, then draw signed scores.

In [ ]:
def build_comparison_obs(
    adata,
    cluster_key,
    group_a_clusters,
    group_b_clusters,
    group_a_label,
    group_b_label,
    new_key="comparison_group"
):
    adata.obs[cluster_key] = adata.obs[cluster_key].astype(str)

    group_a_clusters = [str(x) for x in group_a_clusters]
    group_b_clusters = [str(x) for x in group_b_clusters]
    selected_clusters = group_a_clusters + group_b_clusters

    adata_sub = adata[adata.obs[cluster_key].isin(selected_clusters)].copy()

    mapping = {}
    mapping.update({c: group_a_label for c in group_a_clusters})
    mapping.update({c: group_b_label for c in group_b_clusters})

    adata_sub.obs[new_key] = adata_sub.obs[cluster_key].map(mapping)
    adata_sub.obs[new_key] = pd.Categorical(
        adata_sub.obs[new_key],
        categories=[group_a_label, group_b_label],
        ordered=True
    )

    return adata_sub, new_key

def get_rank_df(
    adata_sub,
    groupby_key,
    target_group,
    reference_group,
    key_added,
    method="wilcoxon",
    pval_adj_cutoff=0.05,
    logfc_min=0.0,
    top_n=8
):
    sc.tl.rank_genes_groups(
        adata_sub,
        groupby=groupby_key,
        groups=[target_group],
        reference=reference_group,
        method=method,
        key_added=key_added
    )

    df = sc.get.rank_genes_groups_df(
        adata_sub,
        group=target_group,
        key=key_added
    )

    df = df[
        (df["pvals_adj"] < pval_adj_cutoff) &
        (df["logfoldchanges"] > logfc_min)
    ].copy()

    df = df.sort_values("scores", ascending=False).head(top_n)

    return df

def make_bidirectional_lollipop(
    df_a,
    df_b,
    group_a_label,
    group_b_label,
    title,
    palette,
    figsize=(5.4, 4.3),
    dpi=300
):
    df_a = df_a.rename(columns={"names": "Protein", "scores": "Score"})
    df_b = df_b.rename(columns={"names": "Protein", "scores": "Score"})

    df_a["Direction"] = group_a_label
    df_b["Direction"] = group_b_label

    df_a["SignedScore"] = df_a["Score"]
    df_b["SignedScore"] = -df_b["Score"]

    df_plot = pd.concat([df_b, df_a], axis=0, ignore_index=True)
    df_plot = df_plot.sort_values("SignedScore", ascending=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    ax.axvline(0, color="#111827", linewidth=1.0)

    for _, row in df_plot.iterrows():
        ax.hlines(
            y=row["Protein"],
            xmin=0,
            xmax=row["SignedScore"],
            color="#D1D5DB",
            linewidth=1.5,
            zorder=1
        )

    ax.scatter(
        df_plot["SignedScore"],
        df_plot["Protein"],
        c=df_plot["Direction"].map(palette),
        s=110,
        edgecolor="black",
        linewidth=0.6,
        zorder=3
    )

    xmax = np.nanmax(np.abs(df_plot["SignedScore"])) * 1.15
    ax.set_xlim(-xmax, xmax)

    sns.despine(ax=ax, top=True, right=True)

    ax.set_title(title, fontsize=12, pad=12, weight="bold")
    ax.set_xlabel("Differential marker score", fontsize=10)
    ax.set_ylabel("Proteins", fontsize=10)
    ax.tick_params(axis="both", labelsize=9)
    ax.grid(axis="x", linestyle="--", alpha=0.35)

    handles = [
        plt.Line2D(
            [0], [0],
            marker="o",
            color="w",
            label=label,
            markerfacecolor=color,
            markeredgecolor="black",
            markersize=8
        )
        for label, color in palette.items()
    ]

    ax.legend(
        handles=handles,
        frameon=False,
        fontsize=8,
        loc="lower right"
    )

    plt.tight_layout()

    return fig, ax, df_plot

### Sacco et al. (2022) — cluster-marker UMAPs

Plot an expression-based UMAP coloured by batch, ProtCell-derived clusters and reference cell types.

In [ ]:
if RUN_SACCO_MARKER_ANALYSIS:
    sacco_cluster_data.obs["Nlouvain_embed"] = pd.Categorical(sacco_cluster_data.obs["Nlouvain_embed"].astype(str))

    sc.pp.neighbors(sacco_cluster_data)
    sc.tl.umap(sacco_cluster_data)
    sacco_cluster_data.uns["ProtCell_marker_umap_graph"] = "expression-based Scanpy neighbour graph"
    sacco_cluster_data.uns["ProtCell_marker_umap_seed"] = 0
    sacco_cluster_data.write_h5ad((SACCO_RESULT_DIR / "marker_representation.h5ad"), compression="gzip")
    axes = sc.pl.umap(sacco_cluster_data, color=["Batch", "Nlouvain_embed", "coarsecelltype"],
                      title=["Batch", "Cluster IDs", "Reference cell type"],
                      legend_loc="right margin", legend_fontsize=8, legend_fontoutline=2,
                      na_in_legend=False, wspace=0.8, show=False)
    fig = axes[0].figure
    fig.savefig(SACCO_FIGURE_DIR / "cluster_marker_umap_triptych.pdf", bbox_inches="tight")
    fig.savefig(SACCO_FIGURE_DIR / "cluster_marker_umap_triptych.png", dpi=300, bbox_inches="tight")
    plt.show()
    for key, label in [("Batch", "batch"), ("Nlouvain_embed", "cluster"), ("coarsecelltype", "celltype")]:
        ax = sc.pl.umap(sacco_cluster_data, color=key, legend_loc="on data" if label == "cluster" else "right margin",
                        legend_fontsize=8, legend_fontoutline=2, show=False)
        fig = ax.figure
        fig.savefig(SACCO_FIGURE_DIR / f"cluster_marker_umap_{label}.pdf", bbox_inches="tight")
        fig.savefig(SACCO_FIGURE_DIR / f"cluster_marker_umap_{label}.png", dpi=300, bbox_inches="tight")
        plt.show()

### Sacco et al. (2022) — cluster protein contrasts

Plot the selected cluster pairs; save the marker tables and PNG/PDF outputs.

In [ ]:
if RUN_SACCO_MARKER_ANALYSIS:
    for group, reference in SACCO_CLUSTER_CONTRASTS:
        counts = sacco_cluster_data.obs["Nlouvain_embed"].value_counts()
        if counts.get(group, 0) < 2 or counts.get(reference, 0) < 2:
            print(f"Contrast {group} vs {reference} unavailable in this representation.")
            continue
        group_label, reference_label = f"Cluster {group}", f"Cluster {reference}"
        comparison, comparison_key = build_comparison_obs(
            sacco_cluster_data, "Nlouvain_embed", [group], [reference], group_label, reference_label)
        forward = get_rank_df(comparison, comparison_key, group_label, reference_label,
                               "rank_forward", method="wilcoxon", pval_adj_cutoff=0.05, logfc_min=0.0, top_n=8)
        reverse = get_rank_df(comparison, comparison_key, reference_label, group_label,
                               "rank_reverse", method="wilcoxon", pval_adj_cutoff=0.05, logfc_min=0.0, top_n=8)
        if len(forward) + len(reverse) == 0:
            print(f"No significant markers for {group} vs {reference}.")
            continue
        fig, ax, marker_table = make_bidirectional_lollipop(
            forward, reverse, group_label, reference_label,
            f"Cluster {group} vs Cluster {reference}",
            palette={group_label: "#2563EB", reference_label: "#F97316"}, figsize=(5.4, 4.3), dpi=300)
        stem = f"diff_marker_cluster{group}_vs_cluster{reference}"
        marker_table.to_csv(SACCO_RESULT_DIR / f"{stem}_markers.csv", index=False)
        fig.savefig(SACCO_FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")
        fig.savefig(SACCO_FIGURE_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        plt.show()

## 2. Beneyto-Calabuig et al. (2023) — load protein measurements

Read the CLR-processed `Dataset107.h5ad` used for this study. `cohort` defines experimental batches/cohorts and `celltype` supplies reference cell types for evaluation. Integration adapts to the full dataset; reference labels determine the evaluation cluster count and assess biological organization.

Outputs for this study are written under `outputs/cluster/Beneyto-Calabuig2023/`.

In [ ]:
AML_OUTPUT_DIR = OUTPUT_DIR / "Beneyto-Calabuig2023"
AML_FIGURE_DIR = AML_OUTPUT_DIR / "figures"
AML_RESULT_DIR = AML_OUTPUT_DIR / "results"
AML_MODEL_DIR = AML_OUTPUT_DIR / "models"
for directory in [AML_FIGURE_DIR, AML_RESULT_DIR, AML_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

aml_data = ad.read_h5ad(DATA_ROOT / "CLR_data/Dataset107.h5ad")
AML_BATCH_COLUMN = "cohort"
AML_CELLTYPE_COLUMN = "celltype"
for column in [AML_BATCH_COLUMN, AML_CELLTYPE_COLUMN]:
    aml_data.obs[column] = aml_data.obs[column].astype(str).replace({"nan": "unknown", "None": "unknown"})
display(aml_data)
display(aml_data.obs.groupby(AML_BATCH_COLUMN, observed=True).size())

RUN_AML_ADAPTATION = True
RUN_AML_EMBEDDINGS = True
RUN_AML_EVALUATION = True
RUN_AML_UMAP_PLOTS = True
# Used when adaptation is disabled; a new fit below automatically selects its new checkpoint.
AML_ADAPTED_MODEL_PATH = PROJECT_ROOT / "checkpoints/BatCorr/107/finetune_5.pt"


### Beneyto-Calabuig et al. (2023) — load pretrained ProtCell

Use the projected CLS output for cell representations. Each study starts from the same pretrained checkpoint.

In [ ]:
aml_model_path = BASE_MODEL_PATH
aml_model = ProteinLanguageModel.load_model_from_checkpoint(str(aml_model_path), device=DEVICE).to(DEVICE)
aml_model.eval()
display(aml_model)

### Beneyto-Calabuig et al. (2023) — fine-tune the integration adapter


In [ ]:
if RUN_AML_ADAPTATION:
    aml_adapter_model = ProteinLanguageModel.load_model_from_checkpoint(
        str(BASE_MODEL_PATH), device=DEVICE
    ).to(DEVICE)
    aml_integration_trainer = BatchCorrectionFineTuner(
        model=aml_adapter_model,
        adata=aml_data,
        device=DEVICE,
        batch_name=AML_BATCH_COLUMN,
        lr=1e-3,
        batch_size=512,
        expr_name="integration",
        trainable_modules={"lora": 1e-3, "reg_proj": 1e-5, "expr_head": 5e-5},
        temperature=0.2,
        reg=0.1,
        reg_m=0.1,
        tau_intra=0.25,
        lambda_intra=500,
    )
    aml_integration_trainer.fit(
        num_epochs=5,
        save=True,
        save_dir=str(AML_MODEL_DIR),
        save_epoch_interval=5,
        early_stop=False,
        lora_rank=16,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["linear1", "linear2", "cls_proj.0", "cls_proj.3"],
    )
    AML_ADAPTED_MODEL_PATH = AML_MODEL_DIR / "integration" / "finetune_5.pt"
print("Adapted checkpoint:", AML_ADAPTED_MODEL_PATH)

### Beneyto-Calabuig et al. (2023) — extract CLS embeddings and assign clusters

Extract zero-shot and adapted ProtCell representations separately. Record the model path and its SHA256 with the saved representation; cluster assignments and UMAP coordinates use that representation.

In [ ]:
aml_representations = {}
if RUN_AML_EMBEDDINGS:
    aml_representation_paths = {'zeroshot': BASE_MODEL_PATH}
    if AML_ADAPTED_MODEL_PATH is not None:
        aml_representation_paths['finetune'] = Path(AML_ADAPTED_MODEL_PATH)
    for (aml_method, aml_checkpoint_path) in aml_representation_paths.items():
        aml_model = ProteinLanguageModel.load_model_from_checkpoint(str(aml_checkpoint_path), device=DEVICE).to(DEVICE)
        aml_model.eval()
        aml_protein_ids = torch.tensor(aml_model.tokenizer.encode(aml_data.var_names.tolist()), device=DEVICE)
        aml_x = aml_data.X.toarray() if sparse.issparse(aml_data.X) else np.asarray(aml_data.X)
        aml_embeddings = []
        with torch.no_grad():
            for aml_start in tqdm(range(0, aml_data.n_obs, 512), desc=aml_method):
                aml_expression = torch.as_tensor(aml_x[aml_start:aml_start + 512], dtype=torch.float32, device=DEVICE)
                aml_ids = aml_protein_ids.unsqueeze(0).expand(len(aml_expression), -1)
                aml_embeddings.append(aml_model.get_cell_embedding(input_ids=aml_ids, expr_values=aml_expression).cpu().numpy())
        aml_result = aml_data.copy()
        aml_result.obsm['X_embed'] = np.vstack(aml_embeddings)
        aml_result = utils.run_leiden(aml_result, n_cluster=aml_result.obs[AML_CELLTYPE_COLUMN].nunique(), use_rep='X_embed', key_added='ProtCell_cluster')
        sc.tl.umap(aml_result, random_state=UMAP_SEED)
        aml_result.uns['ProtCell_checkpoint'] = str(aml_checkpoint_path)
        aml_result.uns['ProtCell_umap_seed'] = UMAP_SEED
        aml_result.uns['ProtCell_dataset_author'] = "Beneyto-Calabuig2023"
        aml_result.uns['ProtCell_checkpoint_sha256'] = hashlib.sha256(Path(aml_checkpoint_path).read_bytes()).hexdigest()
        aml_result.write_h5ad(AML_RESULT_DIR / f'{aml_method}.h5ad', compression='gzip')
        aml_representations[aml_method] = aml_result

### Beneyto-Calabuig et al. (2023) — UMAP by batch, cluster and reference cell type

The three colourings share the same saved coordinates. Save a triptych and individual maps; the cluster map labels cluster IDs on the point cloud.

In [ ]:
if RUN_AML_UMAP_PLOTS:
    assert RUN_AML_EMBEDDINGS and aml_representations, "Run CLS extraction before plotting UMAPs."
    for aml_method, aml_representation in aml_representations.items():
        aml_map_data = aml_representation.copy()
        assert 'X_umap' in aml_map_data.obsm
        aml_map_data.obs['ProtCell_cluster'] = pd.Categorical(aml_map_data.obs['ProtCell_cluster'].astype(str))
        aml_coordinates = aml_map_data.obsm['X_umap'].copy()
        aml_axes = sc.pl.umap(aml_map_data, color=[AML_BATCH_COLUMN, 'ProtCell_cluster', AML_CELLTYPE_COLUMN], title=['Batch', 'ProtCell clusters', 'Reference cell type'], legend_loc='right margin', legend_fontsize=8, legend_fontoutline=2, na_in_legend=False, wspace=0.8, show=False)
        aml_fig = aml_axes[0].figure
        aml_fig.savefig(AML_FIGURE_DIR / f'{aml_method}_umap_triptych.pdf', bbox_inches='tight')
        aml_fig.savefig(AML_FIGURE_DIR / f'{aml_method}_umap_triptych.png', dpi=300, bbox_inches='tight')
        plt.show()
        for (aml_key, aml_label) in [(AML_BATCH_COLUMN, 'batch'), ('ProtCell_cluster', 'cluster'), (AML_CELLTYPE_COLUMN, 'celltype')]:
            aml_ax = sc.pl.umap(aml_map_data, color=aml_key, title=f'ProtCell {aml_method}: {aml_label}', legend_loc='on data' if aml_label == 'cluster' else 'right margin', legend_fontsize=8, legend_fontoutline=2, na_in_legend=False, show=False)
            aml_fig = aml_ax.figure
            aml_fig.savefig(AML_FIGURE_DIR / f'{aml_method}_umap_{aml_label}.pdf', bbox_inches='tight')
            aml_fig.savefig(AML_FIGURE_DIR / f'{aml_method}_umap_{aml_label}.png', dpi=300, bbox_inches='tight')
            plt.show()
        assert np.array_equal(aml_map_data.obsm['X_umap'], aml_coordinates)

### Beneyto-Calabuig et al. (2023) — evaluate cell organization and batch mixing

Calculate ARI, NMI, raw cluster silhouette, scaled biological ASW, batch ASW, kBET, iLISI and cLISI. Compute the summary scores as follows: `avgBio=(ARI+NMI+Bio_ASW)/3`, `avgBatch=(iLISI+Batch_ASW)/2`, and `overall=0.6*avgBio+0.4*avgBatch`. The scaled kBET score is higher for better mixing; dependency failures are recorded separately.

In [ ]:
aml_all_metrics = None
if RUN_AML_EVALUATION:
    assert RUN_AML_EMBEDDINGS and aml_representations, "Run CLS extraction before evaluation."
    aml_metric_rows = []
    aml_metric_status = []
    for aml_method, aml_representation in aml_representations.items():
        aml_evaluation_data = aml_representation.copy()
        aml_expected_checkpoint = BASE_MODEL_PATH if aml_method == "zeroshot" else AML_ADAPTED_MODEL_PATH
        assert Path(aml_evaluation_data.uns["ProtCell_checkpoint"]).resolve() == Path(aml_expected_checkpoint).resolve(), "Rerun embedding extraction after changing the checkpoint."
        assert aml_evaluation_data.uns["ProtCell_checkpoint_sha256"] == hashlib.sha256(Path(aml_expected_checkpoint).read_bytes()).hexdigest(), "Rerun embedding extraction after retraining."
        print(f'Evaluating {aml_method}: {aml_evaluation_data.n_obs:,} cells')
        aml_ari = adjusted_rand_score(aml_evaluation_data.obs[AML_CELLTYPE_COLUMN], aml_evaluation_data.obs['ProtCell_cluster'])
        aml_nmi = normalized_mutual_info_score(aml_evaluation_data.obs[AML_CELLTYPE_COLUMN], aml_evaluation_data.obs['ProtCell_cluster'])
        aml_cluster_silhouette = silhouette_score(aml_evaluation_data.obsm['X_embed'], aml_evaluation_data.obs['ProtCell_cluster'].astype(str))
        aml_bio_asw = scib.me.silhouette(aml_evaluation_data, group_key=AML_CELLTYPE_COLUMN, embed='X_embed')
        aml_batch_asw = scib.me.silhouette_batch(aml_evaluation_data, batch_key=AML_BATCH_COLUMN, group_key=AML_CELLTYPE_COLUMN, embed='X_embed')
        try:
            aml_kbet = scib.me.kBET(aml_evaluation_data, batch_key=AML_BATCH_COLUMN, label_key=AML_CELLTYPE_COLUMN, type_='embed', embed='X_embed')
            aml_metric_status.append(dict(method=aml_method, metric='kBET', status='computed', detail=''))
        except (ModuleNotFoundError, ImportError, RuntimeError) as error:
            aml_kbet = np.nan
            aml_metric_status.append(dict(method=aml_method, metric='kBET', status='unavailable', detail=str(error)))
            print('kBET could not be computed:', error)
        aml_ilisi = scib.me.ilisi_graph(aml_evaluation_data, batch_key=AML_BATCH_COLUMN, type_='embed', use_rep='X_embed')
        aml_clisi = scib.me.clisi_graph(aml_evaluation_data, label_key=AML_CELLTYPE_COLUMN, type_='embed', use_rep='X_embed')
        aml_avgbio = (aml_ari + aml_nmi + aml_bio_asw) / 3
        aml_avgbatch = (aml_ilisi + aml_batch_asw) / 2
        aml_metric_rows.append(dict(method=aml_method, ARI=aml_ari, NMI=aml_nmi, Silhouette=aml_cluster_silhouette, kBET=aml_kbet, iLISI=aml_ilisi, cLISI=aml_clisi, Bio_ASW=aml_bio_asw, Batch_ASW=aml_batch_asw, avgBio=aml_avgbio, avgBatch=aml_avgbatch, overall=0.6 * aml_avgbio + 0.4 * aml_avgbatch))
    aml_all_metrics = pd.DataFrame(aml_metric_rows)
    aml_all_metrics.to_csv(AML_RESULT_DIR / 'metrics.csv', index=False)
    pd.DataFrame(aml_metric_status).to_csv(AML_RESULT_DIR / 'metric_status.csv', index=False)
    display(aml_all_metrics)

### Beneyto-Calabuig et al. (2023) — display all evaluation metrics

Display the complete table calculated above. The current cell representations and metrics stay in memory.


In [ ]:
if RUN_AML_EVALUATION:
    assert aml_all_metrics is not None, "Run the evaluation cell before displaying metrics."
    aml_protcell_metrics = aml_all_metrics.copy()
    aml_metric_columns = ['ARI', 'NMI', 'Silhouette', 'kBET', 'iLISI', 'cLISI', 'Bio_ASW', 'Batch_ASW', 'avgBio', 'avgBatch', 'overall']
    display(aml_protcell_metrics[['method'] + aml_metric_columns])
else:
    print("Enable RUN_AML_EVALUATION to calculate this study's metrics.")


## Release temporary files

Keep the final tables, figures and trained models; delete this notebook’s temporary plotting/JIT caches.

In [ ]:
plt.close("all")
plot_cache.cleanup()
for key, previous in old_cache_settings.items():
    if previous is None:
        os.environ.pop(key, None)
    else:
        os.environ[key] = previous
print("Temporary caches removed.")